## 1) Environment Setup
Mount Google Drive and install required libraries.

In [1]:
!nvidia-smi

Thu Apr  2 14:44:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q transformers datasets evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00


## 2) Data Extraction and Sampling
Extract raw data, refresh destination folders, and build train/dev/test samples.

In [4]:
import os

file_path = '/content/drive/MyDrive/Baginda/liputan6_data.tar.gz'
local_extract_path = '/content/temp_extract'

# Create local directory
os.makedirs(local_extract_path, exist_ok=True)

print(f"Extracting 'canonical' folder to local storage {local_extract_path}...")
# Extract only the canonical directory from the archive
!tar -xzf "{file_path}" -C "{local_extract_path}" liputan6_data/canonical
print("Local extraction finished.")

Extracting 'canonical' folder to local storage /content/temp_extract...
Local extraction finished.


In [5]:
import shutil
import os

old_extract_path = '/content/drive/MyDrive/Baginda/liputan6_data'

print(f"Deleting old extraction folder at {old_extract_path}...")
print("This might take a moment depending on the number of files already extracted.")

if os.path.exists(old_extract_path):
    shutil.rmtree(old_extract_path)
    print("Old folder successfully deleted.")
else:
    print("Old folder does not exist.")

# Recreate the base directory
os.makedirs(old_extract_path, exist_ok=True)
print(f"Created fresh directory at {old_extract_path}")

Deleting old extraction folder at /content/drive/MyDrive/Baginda/liputan6_data...
This might take a moment depending on the number of files already extracted.
Old folder successfully deleted.
Created fresh directory at /content/drive/MyDrive/Baginda/liputan6_data


In [6]:
import os
import random
import shutil
from pathlib import Path

# Configuration
TOTAL_FILES_TO_SAMPLE = 25000
train_ratio = 0.70
dev_ratio = 0.10
test_ratio = 0.20

# Paths
local_base_path = '/content/temp_extract/liputan6_data/canonical'
drive_extract_path = '/content/drive/MyDrive/Baginda/liputan6_data/canonical'

# Calculate counts
train_count = int(TOTAL_FILES_TO_SAMPLE * train_ratio)
dev_count = int(TOTAL_FILES_TO_SAMPLE * dev_ratio)
test_count = TOTAL_FILES_TO_SAMPLE - train_count - dev_count # Ensure it adds up exactly

splits = {
    'train': train_count,
    'dev': dev_count,
    'test': test_count
}

print(f"Target distribution -> Train: {train_count}, Dev: {dev_count}, Test: {test_count}\n")

for split, count in splits.items():
    src_dir = os.path.join(local_base_path, split)
    dst_dir = os.path.join(drive_extract_path, split)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(src_dir):
        # Get all json files in the source directory
        all_files = [f for f in os.listdir(src_dir) if f.endswith('.json')]
        print(f"Found {len(all_files)} files in local {split} folder.")

        # Sample files (or take all if count > available files)
        files_to_copy = random.sample(all_files, min(count, len(all_files)))

        print(f"Copying {len(files_to_copy)} files to Drive {split} folder...")
        for f in files_to_copy:
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
        print(f"Finished copying {split}.\n")
    else:
        print(f"Warning: Source directory {src_dir} not found!")

print("Sampling and copying process completed!")

Target distribution -> Train: 17500, Dev: 2500, Test: 5000

Found 193883 files in local train folder.
Copying 17500 files to Drive train folder...
Finished copying train.

Found 10972 files in local dev folder.
Copying 2500 files to Drive dev folder...
Finished copying dev.

Found 10972 files in local test folder.
Copying 5000 files to Drive test folder...
Finished copying test.

Sampling and copying process completed!


## 3) Data Sanity Check
Inspect one sample file to verify schema and content shape.

In [7]:
import os
import json

sample_dir = '/content/drive/MyDrive/Baginda/liputan6_data/canonical/train'

if os.path.exists(sample_dir):
    sample_files = [f for f in os.listdir(sample_dir) if f.endswith('.json')]
    if sample_files:
        sample_file_path = os.path.join(sample_dir, sample_files[0])
        print(f"Reading sample file: {sample_file_path}\n")

        with open(sample_file_path, 'r') as f:
            data = json.load(f)

        print("JSON Keys:", list(data.keys()))
        print("\nFull JSON Content:")
        print(json.dumps(data, indent=2))
    else:
        print("No JSON files found in the directory.")
else:
    print(f"Directory not found: {sample_dir}")

Reading sample file: /content/drive/MyDrive/Baginda/liputan6_data/canonical/train/222241.json

JSON Keys: ['id', 'url', 'clean_article', 'clean_summary', 'extractive_summary']

Full JSON Content:
{
  "id": 222241,
  "url": "https://www.liputan6.com/news/read/222241/ully-sigar-rusady-bakal-tampil-di-sarajevo",
  "clean_article": [
    [
      "ULLY",
      "Sigar",
      "Rusady",
      "akan",
      "mewakili",
      "Indonesia",
      "dalam",
      "festival",
      "musik",
      "etnik",
      "yang",
      "digelar",
      "di",
      "Sarajevo",
      ",",
      "Bosnia",
      ",",
      "September",
      "mendatang",
      "."
    ],
    [
      "Rencananya",
      ",",
      "kakak",
      "kandung",
      "artis",
      "cantik",
      "Paramitha",
      "Rusady",
      "ini",
      "membawakan",
      "lagu",
      "Musim",
      "Tanam",
      "."
    ],
    [
      "Hal",
      "itu",
      "diungkapkan",
      "Ully",
      "usai",
      "tampil",
      "di",
      "Gedu

## 4) Load and Structure Dataset
Load JSON files into pandas DataFrames for each split.

In [8]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm

def load_data_split(split_path):
    records = []
    if not os.path.exists(split_path):
        print(f"Path not found: {split_path}")
        return pd.DataFrame()

    files = [f for f in os.listdir(split_path) if f.endswith('.json')]
    for f in tqdm(files, desc=f"Loading {os.path.basename(split_path)}"):
        with open(os.path.join(split_path, f), 'r') as file:
            data = json.load(file)

            # Form the full text from nested lists
            article = " ".join([" ".join(sentence) for sentence in data.get('clean_article', [])])
            summary = " ".join([" ".join(sentence) for sentence in data.get('clean_summary', [])])

            records.append({
                'id': data.get('id'),
                'url': data.get('url'),
                'article': article,
                'summary': summary,
                'extractive_summary': data.get('extractive_summary')
            })

    return pd.DataFrame(records)

# Base directory where your sampled data is stored
base_dir = '/content/drive/MyDrive/Baginda/liputan6_data/canonical'

# Load the datasets
print("Loading datasets into Pandas DataFrames...")
df_train = load_data_split(os.path.join(base_dir, 'train'))
df_dev = load_data_split(os.path.join(base_dir, 'dev'))
df_test = load_data_split(os.path.join(base_dir, 'test'))

print("\nData Loading Complete!")
print(f"Train size: {len(df_train)}")
print(f"Dev size: {len(df_dev)}")
print(f"Test size: {len(df_test)}")

# Preview the train dataframe
display(df_train.head())

Loading datasets into Pandas DataFrames...


Loading train:   0%|          | 0/17500 [00:00<?, ?it/s]

Loading dev:   0%|          | 0/2500 [00:00<?, ?it/s]

Loading test:   0%|          | 0/5000 [00:00<?, ?it/s]


Data Loading Complete!
Train size: 17500
Dev size: 2500
Test size: 5000


,id,url,article,summary,extractive_summary
0,222241,https://www.liputan6.com/news/read/222241/ully...,ULLY Sigar Rusady akan mewakili Indonesia dala...,ULLY Sigar Rusady akan mewakili Indonesia dala...,"[0, 2, 1]"
1,38482,https://www.liputan6.com/news/read/38482/dibek...,"Liputan6 . com , Jakarta : Kepolisian Daerah M...",Kasus penipuan berkedok jasa pemasangan iklan ...,"[0, 2]"
2,142818,https://www.liputan6.com/news/read/142818/poli...,"Liputan6 . com , Jakarta : Yusron alias Mahfud...",Polisi kini mengarahkan perburuan terhadap Abu...,"[0, 1]"
3,202997,https://www.liputan6.com/news/read/202997/scha...,"Sehari sebelum debut PD 2006 , situasi Togo se...","Sehari sebelum debut PD 2006 , situasi Togo se...","[0, 1, 11, 10]"
4,293676,https://www.liputan6.com/news/read/293676/ussy...,"Liputan6 . com , Jakarta : Artis Ussy Sulistia...",Ussy Sulistiawaty mengaku sedikit mengerem keg...,"[0, 1]"


## 5) Tokenization and Preprocessing
Convert DataFrames to Hugging Face datasets and tokenize inputs/targets.

In [9]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Convert Pandas DataFrames to Hugging Face Datasets
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_dev),
    "test": Dataset.from_pandas(df_test)
})

# We use a BERT2BERT model tailored for Indonesian text summarization
model_checkpoint = "cahya/bert2bert-indonesian-summarization"

print(f"Loading tokenizer from {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Set max lengths for the articles and the summaries
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["article"]
    targets = examples["summary"]

    # Tokenize the input articles
    model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

    # Tokenize the target summaries
    labels = tokenizer(targets, max_length=max_target_length, padding="max_length", truncation=True)

    # Replace the padding token id with -100 so they are ignored by the loss function
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets... This might take a minute.")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
print("\nTokenization complete!")

# Preview the keys of the first tokenized train record to ensure it contains 'input_ids' and 'labels'
print("\nFeatures in our tokenized dataset:", list(tokenized_datasets["train"][0].keys()))

Loading tokenizer from cahya/bert2bert-indonesian-summarization...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizing datasets... This might take a minute.


Map:   0%|          | 0/17500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


Tokenization complete!

Features in our tokenized dataset: ['id', 'url', 'article', 'summary', 'extractive_summary', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']


## 6) Model and Trainer Setup
Initialize the seq2seq model, metrics, and training arguments.

In [10]:
from transformers import EncoderDecoderModel, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate
import numpy as np

# 1. Load the model
print(f"Loading model {model_checkpoint}...")
model = EncoderDecoderModel.from_pretrained(model_checkpoint)
model.config.tie_word_embeddings = False

# Set model configuration for sequence-to-sequence generation
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size

# Set generation parameters
model.config.max_length = 128
model.config.min_length = 16
model.config.no_repeat_ngram_size = 3
model.config.early_stopping = True
model.config.length_penalty = 2.0
model.config.num_beams = 4

# 2. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 3. Metric (ROUGE)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split()) for label in decoded_labels]

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Extract a few results
    result = {key: value * 100 for key, value in result.items()}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# 4. Training Arguments
batch_size = 8

args = Seq2SeqTrainingArguments(
    output_dir="/content/bert2bert-indonesian-summarization-finetuned",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True, # Enable mixed precision for faster training on GPU
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer setup complete! Ready to start training.")

Loading model cahya/bert2bert-indonesian-summarization...


pytorch_model.bin:   0%|          | 0.00/999M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/999M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/523 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
EncoderDecoderModel LOAD REPORT from: cahya/bert2bert-indonesian-summarization
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
decoder.bert.embeddings.position_ids | UNEXPECTED |  | 
encoder.embeddings.position_ids      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

Trainer setup complete! Ready to start training.


In [22]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

scores = scorer.score(
    target="Liputan6 . com , Jakarta : Artis Ussy Sulistiawaty mengaku sedikit mengerem kegiatan di bulan suci Ramadan . Tujuannya demi menjaga kondisi tubuh yang belakangan rentan terhadap penyakit . \" Rada dikurangi saja , sekarang lebih fokus sama rekaman buat album ketiga , \"ujar Ussy di Hot Shot SCTV , Ahad ( 29/8 ) . Ussy saat ini berada pada kondisi 90 persen fit . Ussy yang pada saat itu ditemani sang kekasih , Andika Pratama berencana meluncurkan album terbaru setelah Lebaran . Pelantun tembang Klik itu berharap deretan lagu barunya mampu memikat hati para pecinta musik Tanah Air . Sebelumnya Ussy dan Andika berkomitmen hanya bertemu setelah berbuka puasa . Itu semua dilakukan demi menghindari hal negatif yang dapat merusak ibadah mereka [ baca : Andhika-Ussy Bicara Soal Makna Puasa ] . ( AIS ) .",
    prediction="Ussy Sulistiawaty mengaku sedikit mengerem kegiatan di bulan Ramadan untuk menjaga kondisi tubuh ."
)

print(scores)

{'rouge1': Score(precision=0.9230769230769231, recall=0.10434782608695652, fmeasure=0.18749999999999997), 'rouge2': Score(precision=0.75, recall=0.07894736842105263, fmeasure=0.14285714285714285), 'rougeL': Score(precision=0.9230769230769231, recall=0.10434782608695652, fmeasure=0.18749999999999997)}


In [27]:
import pandas as pd
from rouge_score import rouge_scorer

# Assuming your dataframe has 'article' and 'summary' columns
# df = pd.read_csv("your_data.csv")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(row):
    scores = scorer.score(
        target=row['article'],
        prediction=row['summary']
    )
    return pd.Series({
        'rouge1': scores['rouge1'].precision,
        'rouge2': scores['rouge2'].precision,
        'rougeL': scores['rougeL'].precision,
    })

rouge_scores = df_train.apply(compute_rouge, axis=1)
df = pd.concat([df_train, rouge_scores], axis=1)

# Aggregate mean across all rows

# print(df[['rouge1', 'rouge2', 'rougeL']].max())

print(df[['rouge1', 'rouge2', 'rougeL']].agg(['min', 'max', 'mean']))

for col in ['rouge1', 'rouge2', 'rougeL']:
    df[f'{col}_bucket'] = pd.cut(
        df[col],
        bins=3,
        labels=['low', 'mid', 'high']
    )
    print(f"\n{col}:")
    print(df[f'{col}_bucket'].value_counts().sort_index())

        rouge1    rouge2    rougeL
min   0.230769  0.000000  0.147059
max   1.000000  1.000000  1.000000
mean  0.856345  0.582448  0.713289

rouge1:
rouge1_bucket
low       126
mid      2980
high    14394
Name: count, dtype: int64

rouge2:
rouge2_bucket
low     3010
mid     8294
high    6196
Name: count, dtype: int64

rougeL:
rougeL_bucket
low     1349
mid     7544
high    8607
Name: count, dtype: int64


## 7) Training and Evaluation Run
Use the trainer in the previous cell to start fine-tuning and then evaluate on the test split.

Suggested next commands:
- `trainer.train()`
- `trainer.evaluate(tokenized_datasets["test"])`